# Pipeline ETL End-to-End: Procesamiento de Vuelos (Arquitectura Medallón)
### Ingesta, Limpieza, Enriquecimiento Meteorológico Dual y Estado Causal de Red

---

## Contexto del Proyecto
Este pipeline implementa una **Arquitectura Medallón** de tres capas para el procesamiento masivo de datos de tráfico aéreo estadounidense durante el año 2023 (~6.57 millones de operaciones):

1.  **Capa Bronce (Raw):**
   - Ingesta de 26 fuentes de datos crudas en formato CSV:
     - `us_flights_2023_*.csv` (12 meses de vuelos comerciales operados).
     - `us_cancelled_div_2023_*.csv` (12 meses de vuelos cancelados o desviados).
     - `airports_geolocation.csv` (metadatos geográficos y nombres oficiales de 364 aeropuertos).
     - `weather_meteo_by_airport.csv` (registros climáticos diarios por aeropuerto: `tavg`, `tmin`, `tmax`, `prcp`, `snow`, `wspd`, `pres`).

2.  **Capa Plata (Cleansed / Conformed):**
   - Limpieza, desduplicación, normalización de tipos y esquemas.
   - Tratamiento riguroso de valores nulos e imputaciones basadas en dominio aeronáutico.
   - Unificación de vuelos operados y cancelados en una estructura columnar consistente.
   - Almacenamiento optimizado en formato **Apache Parquet** comprimido (`snappy`), reduciendo el consumo de almacenamiento y memoria en más de un 80%.

3.  **Capa Oro (Aggregated / Business & Causal Aviation ML Features):**
   - Tablas analíticas de negocio pre-calculadas para lectura en milisegundos en el Centro de Control Ejecutivo (`Streamlit`).
   - Matriz enriquecida de características (`ml_features_dataset.parquet`, **300,000 vuelos x 58 variables base**) con **estado causal rezagado ($s - 1$) de la red aeroportuaria, aerolínea y rotación intradía de la aeronave (`Tail_Number`)**, además de meteorología sincronizada en ambos extremos de la ruta.


In [ ]:
import os
import sys
import glob
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings('ignore')

# Configurar ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import config
config.ensure_dirs()

print("=" * 70)
print(f"Entorno configurado. Raíz del proyecto: {PROJECT_ROOT}")
print(f"Directorio Bronce: {config.BRONZE_DIR}")
print(f"Directorio Plata:  {config.SILVER_DIR}")
print(f"Directorio Oro:    {config.GOLD_DIR}")
print("=" * 70)


## 1. Inspección de Archivos en Capa Bronce (Raw Data)
Verificamos la existencia e integridad de las 26 fuentes de datos crudas antes de iniciar las transformaciones.


In [ ]:
# Listar fuentes disponibles
bronze_airports_geo = config.BRONZE_AIRPORTS / "airports_geolocation.csv"
bronze_weather = config.BRONZE_AIRPORTS / "weather_meteo_by_airport.csv"

flight_files = sorted(glob.glob(str(config.BRONZE_FLIGHTS / "*.csv")))
cancelled_files = sorted(glob.glob(str(config.BRONZE_CANCELLED / "*.csv")))

print(f" Archivos de vuelos encontrados:      {len(flight_files)}")
print(f" Archivos de cancelaciones/desvíos:   {len(cancelled_files)}")
print(f" Archivo de geolocalización presente: {bronze_airports_geo.exists()}")
print(f" Archivo meteorológico presente:      {bronze_weather.exists()}")


## 2. Procesamiento de Metadatos: Aeropuertos y Clima (Bronce -> Plata)
1. **Aeropuertos (`airports_geolocation.csv`)**: Estandarización IATA y exportación a `data/silver/airports.parquet`.
2. **Meteorología (`weather_meteo_by_airport.csv`)**: Imputación jerárquica por estación aeroportuaria y exportación a `data/silver/weather.parquet`.


In [ ]:
# 2.1 Procesar Aeropuertos
print("Procesando aeropuertos (Bronce -> Plata)...")
df_airports = pd.read_csv(bronze_airports_geo)
df_airports['IATA_CODE'] = df_airports['IATA_CODE'].astype(str).str.strip().str.upper()
df_airports.drop_duplicates(subset=['IATA_CODE'], inplace=True)
df_airports.to_parquet(config.SILVER_AIRPORTS, index=False, engine='pyarrow')
print(f"-> Aeropuertos guardados en Plata: {len(df_airports)} aeropuertos únicos.")

# 2.2 Procesar Clima
print()
print("Procesando datos meteorológicos (Bronce -> Plata)...")
df_weather = pd.read_csv(bronze_weather)
df_weather['time'] = pd.to_datetime(df_weather['time'])
df_weather['airport_id'] = df_weather['airport_id'].astype(str).str.strip().str.upper()

df_weather['prcp'] = df_weather['prcp'].fillna(0.0).astype('float32')
df_weather['snow'] = df_weather['snow'].fillna(0.0).astype('float32')

for col in ['tavg', 'tmin', 'tmax', 'wspd', 'pres']:
    if col in df_weather.columns:
        df_weather[col] = df_weather.groupby('airport_id')[col].transform(lambda x: x.fillna(x.median()))
        df_weather[col] = df_weather[col].fillna(df_weather[col].median()).astype('float32')

df_weather.to_parquet(config.SILVER_WEATHER, index=False, engine='pyarrow', compression='snappy')
print(f"-> Registros de clima guardados en Plata: {len(df_weather):,} observaciones.")


## 3. Procesamiento y Limpieza de Vuelos y Cancelaciones (Bronce -> Plata)
Procesamos mes a mes los ~6.57M de vuelos, unificando vuelos operados y cancelados/desviados en particiones `data/silver/flights/flights_YYYY_MM.parquet`.


In [ ]:
def clean_and_unify_monthly_flights(flight_csv_path, cancelled_csv_path=None):
    """Limpia, tipa y fusiona los vuelos de un mes específico."""
    df_f = pd.read_csv(flight_csv_path, low_memory=False)
    df_f['Cancelled'] = 0
    df_f['Diverted'] = 0
    
    if cancelled_csv_path and Path(cancelled_csv_path).exists():
        df_c = pd.read_csv(cancelled_csv_path, low_memory=False)
        for col in ['Manufacturer', 'Model', 'Aircraft_age']:
            if col not in df_c.columns:
                df_c[col] = 'UNKNOWN' if col != 'Aircraft_age' else np.nan
        df_merged = pd.concat([df_f, df_c], ignore_index=True)
    else:
        df_merged = df_f

    df_merged['FlightDate'] = pd.to_datetime(df_merged['FlightDate'])
    df_merged['Day_Of_Week'] = df_merged['Day_Of_Week'].fillna(df_merged['FlightDate'].dt.day_of_week + 1).astype('int8')
    df_merged['Cancelled'] = df_merged['Cancelled'].fillna(0).astype('int8')
    df_merged['Diverted'] = df_merged['Diverted'].fillna(0).astype('int8')

    metric_cols = ['Dep_Delay', 'Arr_Delay', 'Flight_Duration',
                   'Delay_Carrier', 'Delay_Weather', 'Delay_NAS', 
                   'Delay_Security', 'Delay_LastAircraft']
    for col in metric_cols:
        if col in df_merged.columns:
            df_merged[col] = pd.to_numeric(df_merged[col], errors='coerce').fillna(0.0).astype('float32')

    text_cols = ['airline', 'dep_airport', 'arr_airport', 'DepTime_label', 
                 'Dep_Delay_Type', 'Arr_Delay_Type', 'Distance_type']
    for col in text_cols:
        if col in df_merged.columns:
            df_merged[col] = df_merged[col].astype(str).str.strip().str.upper()

    return df_merged

total_rows_silver = 0
for f_path in flight_files:
    file_name = Path(f_path).name
    month_suffix = file_name.replace("us_flights_", "").replace(".csv", "")
    c_path = config.BRONZE_CANCELLED / f"us_cancelled_div_{month_suffix}.csv"
    
    df_month = clean_and_unify_monthly_flights(f_path, c_path if c_path.exists() else None)
    out_parquet = config.SILVER_FLIGHTS / f"flights_{month_suffix}.parquet"
    
    df_month.to_parquet(out_parquet, index=False, engine='pyarrow', compression='snappy')
    total_rows_silver += len(df_month)
    print(f" Mes {month_suffix} procesado: {len(df_month):,} filas -> {out_parquet.name}")

print()
print(f"Total consolidado en Capa Plata: {total_rows_silver:,} registros.")


## 4. Construcción de la Capa Oro (Business Aggregations & Metrics)
Generamos las 5 tablas analíticas agregadas para el dashboard ejecutivo.


In [5]:
silver_parquet_files = sorted(glob.glob(str(config.SILVER_FLIGHTS / "*.parquet")))
print(f"Agregando métricas de {len(silver_parquet_files)} particiones de Plata...")

df_airports_silver = pd.read_parquet(config.SILVER_AIRPORTS)
df_weather_silver = pd.read_parquet(config.SILVER_WEATHER)

daily_list = []
airline_list = []
route_list = []
causes_list = []

for p_file in silver_parquet_files:
    chunk = pd.read_parquet(p_file)
    
    d_agg = chunk.groupby(['FlightDate', 'dep_airport']).agg(
        total_flights=('FlightDate', 'count'),
        cancelled_flights=('Cancelled', 'sum'),
        diverted_flights=('Diverted', 'sum'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        max_dep_delay=('Dep_Delay', 'max'),
        delayed_15m=('Dep_Delay', lambda x: (x >= 15).sum())
    ).reset_index()
    daily_list.append(d_agg)
    
    chunk['YearMonth'] = chunk['FlightDate'].dt.to_period('M').astype(str)
    a_agg = chunk.groupby(['YearMonth', 'airline']).agg(
        total_flights=('FlightDate', 'count'),
        cancelled_flights=('Cancelled', 'sum'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        avg_arr_delay=('Arr_Delay', 'mean'),
        delayed_15m=('Dep_Delay', lambda x: (x >= 15).sum())
    ).reset_index()
    airline_list.append(a_agg)
    
    r_agg = chunk.groupby(['dep_airport', 'arr_airport']).agg(
        flight_count=('FlightDate', 'count'),
        avg_duration=('Flight_Duration', 'mean'),
        avg_dep_delay=('Dep_Delay', 'mean'),
        cancellation_rate=('Cancelled', 'mean')
    ).reset_index()
    route_list.append(r_agg)
    
    c_agg = chunk[chunk['Dep_Delay'] > 0].groupby('YearMonth').agg(
        delay_carrier=('Delay_Carrier', 'sum'),
        delay_weather=('Delay_Weather', 'sum'),
        delay_nas=('Delay_NAS', 'sum'),
        delay_security=('Delay_Security', 'sum'),
        delay_last_aircraft=('Delay_LastAircraft', 'sum')
    ).reset_index()
    causes_list.append(c_agg)

df_gold_daily = pd.concat(daily_list, ignore_index=True)
df_gold_daily = df_gold_daily.merge(
    df_weather_silver,
    left_on=['FlightDate', 'dep_airport'],
    right_on=['time', 'airport_id'],
    how='left'
).drop(columns=['time', 'airport_id'], errors='ignore')
df_gold_daily.to_parquet(config.GOLD_DAILY_AIRPORT, index=False)
print(f" Oro Guardado: {config.GOLD_DAILY_AIRPORT.name}")

df_gold_airports = df_gold_daily.groupby('dep_airport').agg(
    total_departures=('total_flights', 'sum'),
    total_cancellations=('cancelled_flights', 'sum'),
    overall_avg_delay=('avg_dep_delay', 'mean'),
    severe_delayed_flights=('delayed_15m', 'sum')
).reset_index()
df_gold_airports['cancellation_rate'] = (df_gold_airports['total_cancellations'] / df_gold_airports['total_departures']) * 100
df_gold_airports = df_gold_airports.merge(df_airports_silver, left_on='dep_airport', right_on='IATA_CODE', how='inner')
df_gold_airports.to_parquet(config.GOLD_AIRPORT_SUMMARY, index=False)
print(f" Oro Guardado: {config.GOLD_AIRPORT_SUMMARY.name}")

df_gold_airline = pd.concat(airline_list, ignore_index=True).groupby(['YearMonth', 'airline']).sum().reset_index()
df_gold_airline['ontime_rate'] = (1 - (df_gold_airline['delayed_15m'] / df_gold_airline['total_flights'])) * 100
df_gold_airline['cancellation_rate'] = (df_gold_airline['cancelled_flights'] / df_gold_airline['total_flights']) * 100
df_gold_airline.to_parquet(config.GOLD_AIRLINE_MONTHLY, index=False)
print(f" Oro Guardado: {config.GOLD_AIRLINE_MONTHLY.name}")

df_gold_routes = pd.concat(route_list, ignore_index=True).groupby(['dep_airport', 'arr_airport']).agg({
    'flight_count': 'sum',
    'avg_duration': 'mean',
    'avg_dep_delay': 'mean',
    'cancellation_rate': 'mean'
}).reset_index()
df_gold_routes.to_parquet(config.GOLD_ROUTES, index=False)
print(f" Oro Guardado: {config.GOLD_ROUTES.name}")

df_gold_causes = pd.concat(causes_list, ignore_index=True).groupby('YearMonth').sum().reset_index()
df_gold_causes.to_parquet(config.GOLD_DELAY_CAUSES, index=False)
print(f" Oro Guardado: {config.GOLD_DELAY_CAUSES.name}")


Generando dataset causal enriquecido para Machine Learning...
 Dataset causal enriquecido guardado: 300,000 filas x 58 columnas -> g:\My Drive\repositorios de proyectos\procesamiento de datos de vuelos\data\gold\ml_features_dataset.parquet


## 5. Preparación del Dataset Causal de Machine Learning (Gold Layer)
En aviación comercial, las disrupciones siguen un proceso autorregresivo de propagación en red:
$$\text{Día }(t-1) \longrightarrow \text{Morning }(s=1) \longrightarrow \text{Afternoon }(s=2) \longrightarrow \text{Evening }(s=3) \longrightarrow \text{Night }(s=4)$$
Calculamos sobre los 6.57 millones de vuelos el **estado estrictamente pasado (bloque horario anterior $s - 1$)** antes de tomar la muestra estratificada de 300,000 vuelos:
1. **Propagación Intradía de la Aeronave (`Tail_Number`):** Demora de salida/llegada de la misma matrícula en su bloque horario previo del mismo día (`tail_sameday_prev_dep_delay`, `tail_sameday_prev_arr_delay`, `tail_sameday_prev_disrupted`) y confiabilidad móvil de los últimos 3 bloques (`tail_roll3_disrupted`).
2. **Estado Operativo Previo del Hub de Salida y Destino (`slot_order - 1`):** Tasa de disrupción, cancelación y demoras por causa FAA (*Weather*, *NAS*, *Late Aircraft*) observadas en el aeropuerto en la franja inmediatamente anterior (`hub_dep_prev_*`, `hub_arr_prev_*`).
3. **Estado de Red de la Aerolínea (`slot_order - 1`):** Presión operativa sistémica de la aerolínea en el bloque horario previo (`air_sys_prev_*`, `air_dep_prev_*`).
4. **Meteorología Dual (Origen + Destino):** Variables térmicas, pluviales, nivales, eólicas y barométricas en ambos aeropuertos.


In [6]:
print("Generando dataset causal enriquecido para Machine Learning...")
ml_sample_list = []
slot_order_map = {'MORNING': 1, 'AFTERNOON': 2, 'EVENING': 3, 'NIGHT': 4}

for p_file in silver_parquet_files:
    c = pd.read_parquet(p_file, columns=[
        'FlightDate', 'Day_Of_Week', 'airline', 'Tail_Number', 'dep_airport', 'arr_airport',
        'DepTime_label', 'Dep_Delay', 'Arr_Delay', 'Cancelled', 'Flight_Duration', 
        'Distance_type', 'Manufacturer', 'Model', 'Aircraft_age',
        'Delay_Weather', 'Delay_NAS', 'Delay_LastAircraft', 'Delay_Carrier'
    ])
    c['DepTime_label'] = c['DepTime_label'].fillna('AFTERNOON').astype(str).str.strip().str.upper()
    c['slot_order'] = c['DepTime_label'].map(slot_order_map).fillna(2).astype('int8')
    c['is_delayed_or_cancelled'] = ((c['Dep_Delay'] >= 15) | (c['Cancelled'] == 1)).astype('int8')
    c['Month'] = c['FlightDate'].dt.month.astype('int8')
    c['Is_Weekend'] = c['Day_Of_Week'].isin([6, 7]).astype('int8')

    # 1. Densidad programada de slots y rotación diaria
    c['dep_slot_flights'] = c.groupby(['FlightDate', 'dep_airport', 'slot_order'])['FlightDate'].transform('count').fillna(1).astype('int16')
    c['arr_slot_flights'] = c.groupby(['FlightDate', 'arr_airport', 'slot_order'])['FlightDate'].transform('count').fillna(1).astype('int16')
    c['dep_daily_flights'] = c.groupby(['FlightDate', 'dep_airport'])['FlightDate'].transform('count').fillna(1).astype('int16')
    c['aircraft_daily_legs'] = c.groupby(['FlightDate', 'Tail_Number'])['FlightDate'].transform('count').fillna(1).clip(upper=15).astype('int8')

    # 2. Estado causal rezagado (s-1) en Hub de Origen
    dep_slot_stats = c.groupby(['dep_airport', 'FlightDate', 'slot_order'], as_index=False).agg(
        slot_disrupt_rate=('is_delayed_or_cancelled', 'mean'),
        slot_avg_delay=('Dep_Delay', 'mean'),
        slot_cancel_rate=('Cancelled', 'mean'),
        slot_wx_delay=('Delay_Weather', 'mean'),
        slot_nas_delay=('Delay_NAS', 'mean'),
        slot_late_ac_delay=('Delay_LastAircraft', 'mean')
    ).sort_values(['dep_airport', 'FlightDate', 'slot_order'])
    dep_slot_stats['hub_dep_prev_disrupt_rate'] = dep_slot_stats.groupby('dep_airport')['slot_disrupt_rate'].shift(1)
    dep_slot_stats['hub_dep_prev_avg_delay'] = dep_slot_stats.groupby('dep_airport')['slot_avg_delay'].shift(1)
    dep_slot_stats['hub_dep_prev_cancel_rate'] = dep_slot_stats.groupby('dep_airport')['slot_cancel_rate'].shift(1)
    dep_slot_stats['hub_dep_prev_wx_delay'] = dep_slot_stats.groupby('dep_airport')['slot_wx_delay'].shift(1)
    dep_slot_stats['hub_dep_prev_nas_delay'] = dep_slot_stats.groupby('dep_airport')['slot_nas_delay'].shift(1)
    dep_slot_stats['hub_dep_prev_late_ac_delay'] = dep_slot_stats.groupby('dep_airport')['slot_late_ac_delay'].shift(1)
    dep_slot_stats['hub_dep_roll3_disrupt_rate'] = dep_slot_stats.groupby('dep_airport')['hub_dep_prev_disrupt_rate'].transform(lambda s: s.rolling(3, min_periods=1).mean())

    # 3. Estado causal rezagado (s-1) en Hub de Destino
    arr_slot_stats = c.groupby(['arr_airport', 'FlightDate', 'slot_order'], as_index=False).agg(
        arr_slot_disrupt_rate=('is_delayed_or_cancelled', 'mean'),
        arr_slot_avg_delay=('Arr_Delay', 'mean'),
        arr_slot_nas_delay=('Delay_NAS', 'mean')
    ).sort_values(['arr_airport', 'FlightDate', 'slot_order'])
    arr_slot_stats['hub_arr_prev_disrupt_rate'] = arr_slot_stats.groupby('arr_airport')['arr_slot_disrupt_rate'].shift(1)
    arr_slot_stats['hub_arr_prev_avg_delay'] = arr_slot_stats.groupby('arr_airport')['arr_slot_avg_delay'].shift(1)
    arr_slot_stats['hub_arr_prev_nas_delay'] = arr_slot_stats.groupby('arr_airport')['arr_slot_nas_delay'].shift(1)

    # 4. Estado causal rezagado (s-1) de Aerolínea (Local y Sistémico)
    air_dep_stats = c.groupby(['airline', 'dep_airport', 'FlightDate', 'slot_order'], as_index=False).agg(
        air_dep_disrupt_rate=('is_delayed_or_cancelled', 'mean'),
        air_dep_avg_delay=('Dep_Delay', 'mean')
    ).sort_values(['airline', 'dep_airport', 'FlightDate', 'slot_order'])
    air_dep_stats['air_dep_prev_disrupt_rate'] = air_dep_stats.groupby(['airline', 'dep_airport'])['air_dep_disrupt_rate'].shift(1)
    air_dep_stats['air_dep_prev_avg_delay'] = air_dep_stats.groupby(['airline', 'dep_airport'])['air_dep_avg_delay'].shift(1)

    air_sys_stats = c.groupby(['airline', 'FlightDate', 'slot_order'], as_index=False).agg(
        air_sys_disrupt_rate=('is_delayed_or_cancelled', 'mean'),
        air_sys_avg_delay=('Dep_Delay', 'mean')
    ).sort_values(['airline', 'FlightDate', 'slot_order'])
    air_sys_stats['air_sys_prev_disrupt_rate'] = air_sys_stats.groupby('airline')['air_sys_disrupt_rate'].shift(1)
    air_sys_stats['air_sys_prev_avg_delay'] = air_sys_stats.groupby('airline')['air_sys_avg_delay'].shift(1)

    # 5. Estado causal rezagado (s-1) de la Aeronave Exacta (Tail_Number)
    tail_slot_stats = c[c['Tail_Number'].notna() & (c['Tail_Number'] != 'UNKNOWN')].groupby(
        ['Tail_Number', 'FlightDate', 'slot_order'], as_index=False
    ).agg(
        tail_slot_delay=('Dep_Delay', 'max'),
        tail_slot_arr_delay=('Arr_Delay', 'max'),
        tail_slot_disrupted=('is_delayed_or_cancelled', 'max')
    ).sort_values(['Tail_Number', 'FlightDate', 'slot_order'])
    tail_slot_stats['tail_sameday_prev_dep_delay'] = tail_slot_stats.groupby(['Tail_Number', 'FlightDate'])['tail_slot_delay'].shift(1)
    tail_slot_stats['tail_sameday_prev_arr_delay'] = tail_slot_stats.groupby(['Tail_Number', 'FlightDate'])['tail_slot_arr_delay'].shift(1)
    tail_slot_stats['tail_sameday_prev_disrupted'] = tail_slot_stats.groupby(['Tail_Number', 'FlightDate'])['tail_slot_disrupted'].shift(1)
    tail_slot_stats['tail_has_sameday_prev'] = tail_slot_stats['tail_sameday_prev_dep_delay'].notna().astype('int8')
    tail_slot_stats['tail_prev_dep_delay'] = tail_slot_stats.groupby('Tail_Number')['tail_slot_delay'].shift(1)
    tail_slot_stats['tail_prev_arr_delay'] = tail_slot_stats.groupby('Tail_Number')['tail_slot_arr_delay'].shift(1)
    tail_slot_stats['tail_prev_disrupted'] = tail_slot_stats.groupby('Tail_Number')['tail_slot_disrupted'].shift(1)
    tail_slot_stats['tail_roll3_disrupted'] = tail_slot_stats.groupby('Tail_Number')['tail_prev_disrupted'].transform(lambda s: s.rolling(3, min_periods=1).mean())

    c = c.merge(dep_slot_stats.drop(columns=['slot_disrupt_rate', 'slot_avg_delay', 'slot_cancel_rate', 'slot_wx_delay', 'slot_nas_delay', 'slot_late_ac_delay']), on=['dep_airport', 'FlightDate', 'slot_order'], how='left')
    c = c.merge(arr_slot_stats.drop(columns=['arr_slot_disrupt_rate', 'arr_slot_avg_delay', 'arr_slot_nas_delay']), on=['arr_airport', 'FlightDate', 'slot_order'], how='left')
    c = c.merge(air_dep_stats.drop(columns=['air_dep_disrupt_rate', 'air_dep_avg_delay']), on=['airline', 'dep_airport', 'FlightDate', 'slot_order'], how='left')
    c = c.merge(air_sys_stats.drop(columns=['air_sys_disrupt_rate', 'air_sys_avg_delay']), on=['airline', 'FlightDate', 'slot_order'], how='left')
    c = c.merge(tail_slot_stats.drop(columns=['tail_slot_delay', 'tail_slot_arr_delay', 'tail_slot_disrupted']), on=['Tail_Number', 'FlightDate', 'slot_order'], how='left')

    ml_sample_list.append(c.sample(n=min(25000, len(c)), random_state=42))

df_ml = pd.concat(ml_sample_list, ignore_index=True)

# Unir meteorología de Origen y Destino
df_w_dedup = df_weather_silver.drop_duplicates(subset=['time', 'airport_id'])
df_ml = df_ml.merge(df_w_dedup[['time', 'airport_id', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd', 'pres']], left_on=['FlightDate', 'dep_airport'], right_on=['time', 'airport_id'], how='left').drop(columns=['time', 'airport_id'], errors='ignore')
df_w_arr = df_w_dedup[['time', 'airport_id', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wspd', 'pres']].rename(columns={'tavg': 'arr_tavg', 'tmin': 'arr_tmin', 'tmax': 'arr_tmax', 'prcp': 'arr_prcp', 'snow': 'arr_snow', 'wspd': 'arr_wspd', 'pres': 'arr_pres'})
df_ml = df_ml.merge(df_w_arr, left_on=['FlightDate', 'arr_airport'], right_on=['time', 'airport_id'], how='left').drop(columns=['time', 'airport_id'], errors='ignore')

df_ml = df_ml.drop(columns=['Delay_Weather', 'Delay_NAS', 'Delay_LastAircraft', 'Delay_Carrier', 'Arr_Delay'], errors='ignore')
gold_ml_features_path = config.GOLD_DIR / "ml_features_dataset.parquet"
df_ml.to_parquet(gold_ml_features_path, index=False, engine='pyarrow', compression='snappy')
print(f" Dataset causal enriquecido guardado: {len(df_ml):,} filas x {df_ml.shape[1]} columnas -> {gold_ml_features_path}")


AUDITORÍA DE LA ARQUITECTURA MEDALLÓN
• Capa Bronce: 26 archivos CSV originales
• Capa Plata:  14 archivos Parquet limpios
• Capa Oro:    6 tablas analíticas consolidadas:
   -> airline_monthly_metrics.parquet     |   0.01 MB
   -> airport_summary.parquet             |   0.04 MB
   -> daily_airport_metrics.parquet       |   1.84 MB
   -> delay_causes_monthly.parquet        |   0.00 MB
   -> ml_features_dataset.parquet         |  17.30 MB
   -> route_metrics.parquet               |   0.15 MB
Pipeline ETL completado satisfactoriamente.


## 6. Verificación y Resumen de la Arquitectura Medallón
Comprobamos el estado y tamaño de todos los artefactos persistidos en disco.


In [ ]:
print("=" * 80)
print("AUDITORÍA DE LA ARQUITECTURA MEDALLÓN")
print("=" * 80)
print(f"• Capa Bronce: {len(list(config.BRONZE_DIR.glob('**/*.csv')))} archivos CSV originales")
print(f"• Capa Plata:  {len(list(config.SILVER_DIR.glob('**/*.parquet')))} archivos Parquet limpios")
print(f"• Capa Oro:    {len(list(config.GOLD_DIR.glob('*.parquet')))} tablas analíticas consolidadas:")
for f in sorted(config.GOLD_DIR.glob("*.parquet")):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"   -> {f.name:<35} | {size_mb:>6.2f} MB")
print("=" * 80)
print("Pipeline ETL completado satisfactoriamente.")
